# 7) Convolutional Neural Networks (CNNs)

> Mainly employed for the image traitement.

- One of the first neural networks to have been considered very useful.

- Nowadays, they have been substituted by LLMs, despite some research still focuses on CNNs.


**Image**: a matrix $\mathbb{R}^{w\times h}$ of pixels (little squares), or equivalently a function $\phi: [0, h]\times [0, w] \subset \mathbb{R}^2 \rightarrow \mathbb{R}$ such that:

- $\phi(x)= 0$ outside each pixel $x \in [0, h]\times [0, w]$;

- $\phi(x) = const. $ on each pixel $x \in [0, h]\times [0, w]$.

An image $\phi : \mathbb{R}^2 \rightarrow \mathbb{R}$ is on the scale of grays.

An image $\phi : \mathbb{R}^2 \rightarrow \mathbb{R}^{3}$ is on the scale of RGB.

An image $\phi : \mathbb{R}^2 \rightarrow \mathbb{R}^{k}$ is on $k$ different channels.


>***Inductive bias 1/3:*** *CNNs are equivariant by translations*, i.e. if the input is translated, the output is translated by the same amount (its representation in the feature map shifts by the exact same amount).
>
>VS *invariance by translations*: in the input image is translated, its output representation is invariant.

>Equivariance preserves how a transformation acts on the output; invariance means that the output does not change at all. CNNs are not invariant by translations, but invariance can be eventually built from equivariance features, such as global average or max pooling.


- **Images invariant by translation:**

Translations on an image $\phi: \mathbb{R}^2 \rightarrow \mathbb{R}$ are expressed by two operators

$$
\tau_{h, \epsilon}, \tau_{w, \epsilon}: \mathbb{R}^{2} \rightarrow \mathbb{R}^{2}
$$

such that

$$
\tau_{h,\epsilon}(x, y) = (x+ \epsilon, y) \quad\text{and}\quad \tau_{w,\epsilon}(x, y) = (x, y+\epsilon).
$$.

We say that an image $\phi$ is invariant by translation if $\phi \circ \tau_{h, \epsilon} = \phi \circ \tau_{w, \epsilon} = \phi$, where $\circ$ is the composition.

- **Layers / Functions invariant for translation:**

We consider images $\phi$ with *compact support* and *smooth*. From a mathematical perspective, this is useful to exploit the functional analysis machinery. In this context, a transformation over an image $\phi$ is a function $F: C^{\infty}_{c}(\mathbb{R}^{2}) \rightarrow C^{\infty}_{c}(\mathbb{R}^{2})$, and the same are also transformations invariant for translations.

A layer $F(x) = \max\left( A x + b, 0\right)$, taking in input an image $\phi$, is invariant by translation, if
$$
F(\phi \circ \tau_{h, \epsilon}) = F(\phi \circ \tau_{w, \epsilon}) = F(\phi).
$$

In theory, this definition works and a translated image on a compact support is still on a compact support. In practice, this is not happening, since the image can be shifted outside its visual area. Therefore, CNNs can not really be invariant over translations. Anyways, CNNs are equivariant for translation, and invariance can be built from equivariance by global average or max pooling.


- **Layers / Functions equivariant for translation:**

We say that a later / function $F$, taking in input an image $\phi$, is equivariant for translation if

$$
F(\phi \circ \tau_{h, \epsilon}) = F(\phi) \circ \tau_{h, \epsilon} \quad \text{and} \quad F(\phi \circ \tau_{w, \epsilon}) = F(\phi) \circ \tau_{w, \epsilon},
$$

i.e. a translation of the input produces a corresponding translation of the output / feature map.


## Architecture of CNNs:

A **filter**:

- reads the image by sovrapposition and performing convolutions, pixel by pixel;

- weights the pixels by its weights (these weights are shared for all the image locations);


> ***Inductive bias 2/3***: *local connectivity*, i.e. the filter reads each pixel "locally" considering its nearby neightbors, because in general nearby pixels are related.

> ***Inductive bias 3/3***: *parameters / weights sharing*, because the model's parameters belong to the filter.

When the filter reads an input image and applies convolution, the feature map / output is shrinked.

--> **Padding**: adds zeros all around the borders of the image, to avoid output to be shriked.

For example, with padding=1 we add all around the image one single border of zeros.

**Rmk)** *CNNs operations on images are invertible*, because convolutions are invertible functions in the mathematical sense.

## imagenet(te):

**imagenet**: the most used dataset of labelled images worldwide, to evaluate new models for the images traitement. 

-> Enourmous dataset, which we will not be able to download due to its size.

-> Requires days of time for a single evaluation, even with GPUs.

## Example of CNN:

In [3]:
import torch

In [4]:
diag = torch.tensor(
    [[1.0, 0, 0],
     [0, 1.0, 0],
     [0, 0, 1.0]],
     requires_grad=False
)

In [ ]:
with torch.no_grad():

    arrow.weight.copy_(diag)

    arrow.bias.copy_(bias)

    y = arrow

In [ ]:
def plot(x, y, figsize=(6, 3), cmap=None):

    _, axes = plt.subplots(1, 2, figsize=figsize)

    axes[0].imshow(x.squeeze(), cmap=cmap)
    axes[0].set_title('Original image')
    axes[1].imshow(np.maximum(y.squeeze(), 0))
    axes[1].set_title('Transformed image')


plot(x, y)

## ResNet

*Without skip connections:* the loss function is generally more complex with many peaks and local minima and flat regions. With 20 arrows/layers in the NN, the surface is already enough complicated.

*With skip connections:* the loss function is generally smooth and eventually with a unique global minimum. New layers connections are defined by $y = f(x) + x$ and called **residual connections**.

> **Skip connections** improves the surface of the loss function and thus, optimizing the weights.

> The idea is "externally" connecting non-successive layers and skipping the within internal successive connections. This lowers the number of parameters to learn.

In [ ]:
# BASIC BLOCK

class BasicBlock(nn.Module):
    def __init__(self, inplanes, planes, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=3, stride=stride, bias=False, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLu(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self.make_resnet_layer(BasicBlock, 64, 64, 3)
        self.layer2 = self.make_resnet_layer(BasicBlock, 64, 128, 4, stride=2)
        self.layer3 = self.make_resnet_layer(BasicBlock, 128, 256, 6, stride=2)
        self.layer4 = self.make_resnet_layer(BasicBlock, 256, 512, 3, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = 

In [ ]:
resnet34 = ResNet(num_classes = 10)

## What are the new elements wrto VGG ?

- residual connections (skip connections): connects non successive layers and skip the connections within.
- global pooling: contract all the pixels into a single one (?).
- batch normalization: normalize the batches.
- bottleneck: 
- dropout: while weights are optimised, deactivate at each run some neurons; to avoid that that the NN relies to much only on specific neurons.